# FinTok · Audio pipeline final

Genera **un WAV por escena** usando F5-TTS.

La decisión de quién habla ya viene del JSON producido por `src/text`.  
`src/audio` **no asigna voces por número de escena**.

Ejemplo:

```json
"speaker": "ANDREA"
```

selecciona la referencia de Andrea, mientras que:

```json
"speaker": "MIRIAM"
```

selecciona la referencia de Miriam.

El valor original de `speaker` se conserva en el JSON de salida.


## 1. Clonar o actualizar FinTok

Pon la URL real de tu repositorio en `REPO_URL`.

In [ ]:
from pathlib import Path

REPO_URL = "PEGA_AQUI_LA_URL_DE_TU_REPO"
REPO_DIR = Path("/content/FinTok")

if (REPO_DIR / ".git").exists():
    print("✓ FinTok ya está clonado. Actualizando...")
    %cd /content/FinTok
    !git pull
    %cd /content
else:
    !git clone "$REPO_URL" /content/FinTok


## 2. Instalar dependencias

In [ ]:
import os

# Evita el error de PYTHONHASHSEED observado en Colab.
os.environ["PYTHONHASHSEED"] = "0"

!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U f5-tts soundfile

print("✓ Dependencias instaladas")


## 3. Comprobar referencias de voz

El pipeline espera estos WAV:

- Andrea: `data/audio_samples/03_EN_ANDREA_SHORT_FAST.wav`
- Miriam: `data/audio_samples/06_EN_MIRIAM_SHORT_FAST.wav`


In [ ]:
from pathlib import Path
from IPython.display import Audio, display

REPO_DIR = Path("/content/FinTok")

references = {
    "ANDREA": REPO_DIR / "data/audio_samples/03_EN_ANDREA_SHORT_FAST.wav",
    "MIRIAM": REPO_DIR / "data/audio_samples/06_EN_MIRIAM_SHORT_FAST.wav",
}

for speaker, path in references.items():
    if not path.exists():
        raise FileNotFoundError(f"No encuentro {speaker}: {path}")

    print(f"✓ {speaker}: {path}")
    display(Audio(filename=str(path)))


## 4. Seleccionar el JSON generado por texto

El JSON debe contener `speaker` en cada escena.

In [ ]:
INPUT_JSON = "/content/FinTok/data/outputs/script.json"

print("JSON de entrada:", INPUT_JSON)


## 5. Revisar qué voz se usará

Esta comprobación se hace **antes de cargar F5-TTS**.

In [ ]:
import json

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    preview = json.load(f)

for scene in preview["scenes"]:
    print(
        f"scene {scene['scene_id']} | "
        f"{scene['scene_type']} | "
        f"speaker={scene['speaker']}"
    )


## 6. Generar los audios

El notebook llama al código real de `src/audio/audio_generator.py`.

In [ ]:
import sys

sys.path.insert(0, "/content/FinTok")

from src.audio.audio_generator import generate_audio_from_json

result = generate_audio_from_json(
    input_json=INPUT_JSON,
    speed=1.0,
)

print("\n✓ Pipeline de audio terminado")


## 7. Escuchar las cuatro escenas

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

repo_root = Path("/content/FinTok")

for scene in result["scenes"]:
    print(
        f"Escena {scene['scene_id']} | "
        f"{scene['scene_type']} | "
        f"{scene['speaker']}"
    )
    print(scene["text"])
    print("Audio:", scene["audio_path"])

    audio_path = Path(scene["audio_path"])
    if not audio_path.is_absolute():
        audio_path = repo_root / audio_path

    display(Audio(filename=str(audio_path)))
    print("-" * 80)


## 8. Ejecución fuera de Colab

Desde la raíz de FinTok:

```bash
python -m src.audio.audio_generator data/outputs/script.json
```

Opcionalmente:

```bash
python -m src.audio.audio_generator data/outputs/script.json \
    --output-dir data/audio_output/AAPL_2024 \
    --output-json data/audio_output/AAPL_2024/script_with_audio.json \
    --speed 1.0
```

El flujo queda:

`src/text → JSON con speaker → src/audio → 1 WAV por scene → JSON con audio_path`
